# VLM Integrated Gradients - Google Colab

This notebook uses Integrated Gradients to explain Vision-Language Model predictions.

**Requirements:** Google Colab with GPU runtime (Runtime > Change runtime type > GPU)

## Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone your GitHub repository
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git
%cd YOUR_REPO_NAME

In [ ]:
# Install numpy first (required specific version)
!pip uninstall -y numpy
!pip install numpy==1.26.4

print("\n⚠️  Please restart runtime: Runtime > Restart runtime")
print("After restart, skip this cell and run the next one.")

In [ ]:
# After restart, install other dependencies
!pip install -q -r requirements.txt

## Step 2: Import Libraries and Load Model

In [ ]:
# Import the modules
import sys
sys.path.insert(0, './src')

from src import load_model, vqa_interpret, temporal_vqa_interpret, text_vqa_interpret
import torch
import gc

print("✓ Imports successful")

In [ ]:
# Load the model
model, processor = load_model()

# Check GPU status
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("\n⚠️  Warning: GPU not available. Please enable GPU in Runtime > Change runtime type")

## Step 3: Upload Image and Run Analysis

In [ ]:
# Clear GPU memory before analysis
gc.collect()
torch.cuda.empty_cache()

print("✓ Memory cleared and ready")

In [ ]:
# Upload your image
from google.colab import files

print("Click 'Choose Files' to upload an image...")
uploaded = files.upload()

if uploaded:
    image_path = list(uploaded.keys())[0]
    print(f"\n✓ Uploaded: {image_path}")
else:
    print("No file uploaded.")

In [ ]:
# Define your questions
questions = [
    "What is in this image?",
    "What color is the main object?"
]

# Run the analysis
vqa_interpret(
    image_path=image_path,
    questions=questions,
    model=model,
    processor=processor,
    show_top_k=10
)

## Optional: Run Another Analysis

You can run the above two cells again to analyze a different image with different questions.

## Step 4: Video Analysis (Optional)

Upload and analyze a video file with temporal processing.

In [ ]:
# Clear GPU memory before video analysis
gc.collect()
torch.cuda.empty_cache()

print("✓ Memory cleared and ready for video analysis")

In [ ]:
# Upload your video
from google.colab import files

print("Click 'Choose Files' to upload a video...")
uploaded_video = files.upload()

if uploaded_video:
    video_path = list(uploaded_video.keys())[0]
    print(f"\n✓ Uploaded: {video_path}")
else:
    print("No video uploaded.")

In [ ]:
# Define questions for video analysis
video_questions = [
    "What is happening?",
    "What object is visible?"
]

# Run temporal analysis
# Parameters:
# - fps_sample: How many frames per second to sample (1 = 1 frame/sec, 0.5 = 1 frame/2 sec)
# - max_frames: Maximum frames to process (None = all sampled frames)
# - show_frame_visualizations: Show heatmap for each frame (slow, use False for long videos)
# - show_timeline: Show timeline graphs of predictions over time
# - save_results: Save reports and visualizations to files

results = temporal_vqa_interpret(
    video_path=video_path,
    questions=video_questions,
    model=model,
    processor=processor,
    fps_sample=1,  # Sample 1 frame per second
    max_frames=20,  # Process max 20 frames (adjust based on video length)
    show_frame_visualizations=False,  # Set True to see individual frame heatmaps
    show_timeline=True,  # Show timeline graphs
    save_results=True  # Save reports to 'video_analysis' folder
)

## Step 5: Text Attribution Analysis (Optional)

Analyze which words in your question influence the prediction, and compare image vs text contribution.

In [ ]:
# Clear GPU memory before text attribution analysis
gc.collect()
torch.cuda.empty_cache()

print("✓ Memory cleared and ready for text attribution analysis")

In [ ]:
# Define questions for text attribution
text_questions = [
    "What color is the cat?",
    "Where is the cat?"
]

text_results = text_vqa_interpret(
    image_path=image_path,
    questions=text_questions,
    model=model,
    processor=processor,
    mode='both',
    n_steps=10,
    show_visualizations=True
)